## build_fact_zillow
Rebuilds `gold.fact_zillow_metro_monthly` from `silver.fact_zillow_metro_monthly`: carries the three source measures (native precision) and **derives** `price_to_rent_ratio` and `gross_rental_yield_pct` (null-safe, rounded 2 dp — Gold rounds computed values). Full rebuild via `INSERT OVERWRITE`. Spec: `gold_layer_design.md` §3.4 / §5.3.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
STEP_SEQUENCE = 4                       # position owned by the orchestrator (G4)
SOURCE_TABLE  = f"{SILVER}.fact_zillow_metro_monthly"
TARGET_TABLE  = f"{GOLD}.fact_zillow_metro_monthly"
DIM_GEO       = f"{GOLD}.dim_geo"
DIM_DATE      = f"{GOLD}.dim_date"

In [ ]:
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "gold",
    target_table    = TARGET_TABLE,
)
print(f"build_fact_zillow: step_log_id={step.step_log_id}")

In [ ]:
# Carry source measures native; derive the two ratios null-safe (guard the denominator to avoid
# ANSI divide-by-zero) and round to 2 dp. Columns are in G0 order for the positional INSERT.
try:
    src = spark.table(SOURCE_TABLE)
    rows_read = src.count()
    rent_ann = F.col("typical_rent") * F.lit(12)
    staged = src.select(
        "geo_key", "date_key",
        "typical_home_value", "typical_rent", "inventory_active",
        F.round(F.when((F.col("typical_rent").isNotNull()) & (F.col("typical_rent") != 0),
                       F.col("typical_home_value") / rent_ann), 2).alias("price_to_rent_ratio"),
        F.round(F.when((F.col("typical_home_value").isNotNull()) & (F.col("typical_home_value") != 0),
                       rent_ann / F.col("typical_home_value") * F.lit(100)), 2).alias("gross_rental_yield_pct"),
        F.current_timestamp().alias("inserted_ts"),
        F.current_timestamp().alias("updated_ts"),
    )
    staged.createOrReplaceTempView("gold_fact_zillow_staging")
    step.rows_read = rows_read
    print(f"build_fact_zillow: read {rows_read:,} Silver rows")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Full rebuild via INSERT OVERWRITE (design §2.2) — preserves G0 schema/PK/FK/COMMENTs.
# FK is informational (not enforced), so check geo_key/date_key resolve in the Gold dims here.
transform_started = datetime.now(timezone.utc)
try:
    staged = spark.table("gold_fact_zillow_staging")
    geo_orphans  = staged.join(spark.table(DIM_GEO).select("geo_key"),  "geo_key",  "left_anti").count()
    date_orphans = staged.join(spark.table(DIM_DATE).select("date_key"), "date_key", "left_anti").count()
    if geo_orphans or date_orphans:
        raise AssertionError(f"[{TARGET_TABLE}] FK orphans: geo={geo_orphans:,} date={date_orphans:,}")

    spark.sql(f"INSERT OVERWRITE TABLE {TARGET_TABLE} SELECT * FROM gold_fact_zillow_staging")

    post_count = spark.table(TARGET_TABLE).count()
    if post_count != step.rows_read:
        raise AssertionError(f"[{TARGET_TABLE}] Row-count mismatch: read {step.rows_read:,}, wrote {post_count:,}.")
    step.rows_written = post_count
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=post_count, rows_inserted=post_count, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"build_fact_zillow: wrote {post_count:,} rows to {TARGET_TABLE}")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        error_message=f"{type(e).__name__}: {e}", ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise